In [ ]:

import ast
import math


def other(player):
    return "MIN" if player == "MAX" else "MAX"


# Convert a path such as (0, 1, 0) into readable text
def label(path):
    return "ROOT" + "".join(
        f" -> child {i + 1}" for i in path
    )


# Find all leaf values inside a subtree
def leaf_values(node):
    if not isinstance(node, list):
        return [node]

    result = []
    for child in node:
        result.extend(leaf_values(child))

    return result


def validate(node):
    if isinstance(node, list):
        if not node:
            raise ValueError("Every node must have at least one child.")

        for child in node:
            validate(child)

    elif type(node) not in (int, float):
        raise ValueError("Use nested lists and numeric leaf values.")


def alpha_beta(tree, root_player="MAX"):

    pruned = []
    visited = 0

    def search(node, depth, alpha, beta, path):
        nonlocal visited

        # If this is a leaf, return its value
        if not isinstance(node, list):
            visited += 1
            print(f"{label(path)}: Visit leaf = {node}")
            return node, path

        # Decide whether the current node is MAX or MIN
        player = (
            root_player if depth % 2 == 0
            else other(root_player)
        )

        print(f"\nEnter {player} at {label(path)}")
        print(f"Alpha = {alpha}, Beta = {beta}")

        if player == "MAX":

            best = -math.inf
            best_path = None

            for i, child in enumerate(node):

                print(f"  Explore child {i + 1}")

                value, candidate = search(
                    child, depth + 1,
                    alpha, beta, path + (i,)
                )

                if best_path is None or value > best:
                    best = value
                    best_path = candidate

                alpha = max(alpha, best)

                print(f"  MAX value = {best}")
                print(f"  Updated alpha = {alpha}")

                # Check cutoff
                if alpha >= beta:

                    print("  CUTOFF: Alpha >= Beta")

                    for j in range(i + 1, len(node)):
                        skipped = path + (j,)
                        values = leaf_values(node[j])

                        pruned.append((skipped, values))

                        print(
                            f"  PRUNED: {label(skipped)}"
                            f" | Leaves = {values}"
                        )

                    break

            return best, best_path

        else:

            best = math.inf
            best_path = None

            for i, child in enumerate(node):

                print(f"  Explore child {i + 1}")

                value, candidate = search(
                    child, depth + 1,
                    alpha, beta, path + (i,)
                )

                if best_path is None or value < best:
                    best = value
                    best_path = candidate

                beta = min(beta, best)

                print(f"  MIN value = {best}")
                print(f"  Updated beta = {beta}")

                # Check cutoff
                if alpha >= beta:

                    print("  CUTOFF: Alpha >= Beta")

                    for j in range(i + 1, len(node)):
                        skipped = path + (j,)
                        values = leaf_values(node[j])

                        pruned.append((skipped, values))

                        print(
                            f"  PRUNED: {label(skipped)}"
                            f" | Leaves = {values}"
                        )

                    break

            return best, best_path

    # Generate a readable optimal path
    def describe_path(path):

        parts = [f"ROOT ({root_player})"]
        current = tree

        for level, index in enumerate(path, start=1):

            child = current[index]

            if isinstance(child, list):
                player = (
                    root_player if level % 2 == 0
                    else other(root_player)
                )
                parts.append(f"child {index + 1} ({player})")
            else:
                parts.append(
                    f"child {index + 1} (leaf={child})"
                )

            current = child

        return " -> ".join(parts)

    # Start searching from the root
    root_value, best_path = search(
        tree, 0, -math.inf, math.inf, ()
    )

    print("\n" + "=" * 50)
    print("FINAL RESULT")
    print("Root value:", root_value)
    print("One optimal path:", describe_path(best_path))
    print("Visited leaves:", visited)

    if pruned:
        print("\nPruned subtrees:")

        for path, values in pruned:
            print(label(path), "->", values)
    else:
        print("Pruned subtrees: None")

    print("=" * 50)


if __name__ == "__main__":

    print("ALPHA-BETA PRUNING")
    print("Enter a tree using nested lists.")
    print("Numbers represent leaf values.")

    raw_tree = input("Tree: ")

    try:
        tree = ast.literal_eval(raw_tree)

        if not isinstance(tree, list):
            raise ValueError("The root must be a list.")

        validate(tree)

        root = input(
            "Root player [MAX/MIN, default MAX]: "
        ).strip().upper() or "MAX"

        if root not in ("MAX", "MIN"):
            raise ValueError("Root must be MAX or MIN.")

        alpha_beta(tree, root)

    except (SyntaxError, ValueError, TypeError) as error:
        print("Invalid input:", error)


ALPHA-BETA PRUNING
Enter a tree using nested lists.
Numbers represent leaf values.
Tree: [[[[5, 6], [7, 4, 5]], [[3]]], [[[6], [6, 9]], [[7]]], [[[5]], [[9, 8], [6]]]]
Root player [MAX/MIN, default MAX]: MAX

Enter MAX at ROOT
Alpha = -inf, Beta = inf
  Explore child 1

Enter MIN at ROOT -> child 1
Alpha = -inf, Beta = inf
  Explore child 1

Enter MAX at ROOT -> child 1 -> child 1
Alpha = -inf, Beta = inf
  Explore child 1

Enter MIN at ROOT -> child 1 -> child 1 -> child 1
Alpha = -inf, Beta = inf
  Explore child 1
ROOT -> child 1 -> child 1 -> child 1 -> child 1: Visit leaf = 5
  MIN value = 5
  Updated beta = 5
  Explore child 2
ROOT -> child 1 -> child 1 -> child 1 -> child 2: Visit leaf = 6
  MIN value = 5
  Updated beta = 5
  MAX value = 5
  Updated alpha = 5
  Explore child 2

Enter MIN at ROOT -> child 1 -> child 1 -> child 2
Alpha = 5, Beta = inf
  Explore child 1
ROOT -> child 1 -> child 1 -> child 2 -> child 1: Visit leaf = 7
  MIN value = 7
  Updated beta = 7
  Explore chil